> Kalo di BERT itu kita ngetraining model pake Mask Lang Modeling gitu misal ada 15% yg di mask, yaudah brati ngitung loss cuma dari 15% token itu. Sisa 85%nya dikemanain? ga diapa2in jadi kek boros compute aja 

Nah si ELECTRA ini lahh yg coba nackle masalah tsb. ELECTRA ngangkat approach `Replaced Token Detection`, instead of cuma nebak token2 yg ke mask/bolong disini electra milih buat as a dunia "game" (mirip GAN) police vs scammer. So ELECTRA itu ada 2 model besar:
- Generator: ini si scammer, jadi dia tu sbnernya juga BERT(small size-nya), tasknya adlaah ngisi `masked token` itu dengan guessingnya (bs bener bs salah)
- Discriminator: ini police-nya, model utamanya -> tasknya ngecek each token di kalimat dan dia `nebak` ini token asli dataset or "cuma" tipuan yang digenerate oleh si generator kita?

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
VOCAB = {
    '[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3,
    'Aku': 4, 'aku': 5, 'sayang': 6, 'banget': 7, 'sama': 8,
    'ibu': 9, 'dia': 10, 'benci': 11, 'anjing': 12, 'liar': 13
}
ID2WORD = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

In [5]:
# Kalimat asli: "Aku sayang banget sama ibu", terus kita mask si sayang (posisi ke-2)
orig_tokens = [1,4,6,7,8,9,2] # [CLS] Aku sayang banget sama ibu [SEP]
masked_tokens = [1,4,3,7,8,9,2]
mlm_labels = [-100,-100,6,-100,-100,-100,-100]

input_ids = torch.tensor([masked_tokens]) # B=1,T=7
labels_mlm = torch.tensor([mlm_labels])
orig_ids = torch.tensor([orig_tokens])


- klo ad yg tanya itu kenapa sii kok -100, umumnya sii itu karna kan ngitung lossnya yg ke mask aja kann, nah itu kan mechansim dibelakangnya pake si `CrossEntropyLoss(ignore_index=-100)`

# Generator & Discriminator Arch

In [ ]:
# encodernya buat simple aja, klo real implement kita pake yg proper yaa (e.g. BERT)
class SimpleEncoder(nn.Module):
    def __init__(self,d_model):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE,d_model) 
        # dummy transformer layer dr torch bawaan
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model,nhead=2,batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer=encoder_layer,num_layers=1)

    def forward(self,x):
        # x nya itu kan (B,T)
        x = self.emb(x)
        out = self.transformer(x)
        return out

## Generator

In [ ]:
class ElectraGenerator(nn.Module):
    def __init__(self,d_model=32):
        super().__init__()
        self.encoder = SimpleEncoder(d_model)
        # MLM head dia yg nebak vocab
        self.mlm_head = nn.Linear(d_model,VOCAB_SIZE) # klo ini model milih vocab sizenya yg keambil di datanya misal Linear(32,1000) means 32 feature with 1000 posibility token

    def forward(self,x):
        features = self.encoder(x) # (B,T,d_model)
        logits = self.mlm_head(x) # (B,T,vocab_size)
        return logits

## Discriminator

In [ ]:
class ElectraDiscriminator(nn.Module):
    def __init__(self,d_model=64):
        super().__init__()
        self.encoder = SimpleEncoder(d_model)
        # RTD Head: Binary Class (1=replaced by gen, 0=real from dt)
        self.rtd_head = nn.Linear(d_model,1)

    def forward(self,x):
        features = self.encoder(x) # (B,T,d_model)
        logits = self.rtd_head(features) # (B,T,1) <- ini 1 karena dia cuma pen tau either is ✅ or ❌
        return logits.squeeze(-1) 